# 条件质心可视化

面向 `trishift / scouter / gears / genepert` 的 condition centroid 级别可视化。

这个 notebook 会完成以下工作：
- 对每个 condition 计算 `DE` 特征空间中的质心
- 比较每个 condition 下 `Truth / Pred / Ctrl` 的 centroid
- 输出 centroid UMAP、delta-centroid UMAP 以及对应统计结果

默认约定如下：
- `trishift/scouter` 默认推荐使用 `variant_tag="nearest"`
- `gears/genepert` 不需要设置 `variant_tag`
- `feature_mode="full"` 需要结果 `pkl` 中包含 full-gene 特征；否则请使用默认的 `deg`


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

repo_root = Path.cwd()
if not (repo_root / "scripts").exists() and (repo_root.parent / "scripts").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import scripts.trishift.analysis.condition_centroid_vis as condition_centroid_vis
importlib.reload(condition_centroid_vis)

run_condition_centroid_visualization = condition_centroid_vis.run_condition_centroid_visualization

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
repo_root


## 参数

在这里设置模型、数据集、split，以及输出目录和 UMAP 作图参数。


In [ ]:
model_name = "trishift"  # 可选: trishift | scouter | gears | genepert
dataset = "adamson"
split_id = 1
result_dir = None
out_root = None
variant_tag = "nearest"  # trishift/scouter 可选: nearest / random / None
feature_mode = "deg"  # 特征模式: deg | full
include_ctrl = True
plot_delta = True
umap_n_neighbors = None
umap_min_dist = 0.15
save_dpi = 420
seed = 24

run_kwargs = dict(
    model_name=model_name,
    dataset=dataset,
    split_id=split_id,
    result_dir=result_dir,
    out_root=out_root,
    variant_tag=variant_tag,
    feature_mode=feature_mode,
    include_ctrl=include_ctrl,
    plot_delta=plot_delta,
    umap_n_neighbors=umap_n_neighbors,
    umap_min_dist=umap_min_dist,
    save_dpi=save_dpi,
    seed=seed,
)
run_kwargs


In [ ]:
result = run_condition_centroid_visualization(**run_kwargs)
print(f"out_dir: {result.out_dir}")
display(result.summary_df)
display(result.metrics_df.head(20))
display(result.points_df.head(20))


## 图像

这里展示生成的 centroid UMAP 和 delta-centroid UMAP 图。


In [ ]:
for key in ["truth_vs_pred_centroid_umap", "truth_pred_ctrl_centroid_umap", "delta_centroid_umap"]:
    img_path = result.figure_paths.get(key)
    if not img_path:
        continue
    print(key, img_path)
    display(Image(filename=img_path, width=1100))


## 输出文件

这里会打印本次运行生成的 CSV、图像和元信息文件路径。


In [ ]:
print(result.out_dir / "condition_centroid_points.csv")
print(result.out_dir / "condition_centroid_metrics.csv")
print(result.out_dir / "condition_centroid_summary.csv")
print(result.out_dir / "run_meta.json")
